In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("..")

import numpy as np
from numpy.linalg import norm
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from tqdm import tqdm
from dataclasses import dataclass
from lunanav.constants import *  # noqa: F403
from lunanav.sim.simulator import SimParams, RigidBody, run_sim, SimResults, reverse_sim_results
from lunanav.sim.sensors import (
    SensorEnvironment, Sensor, SensorSuite, SensorName,
    accelerometer_sensor, gyroscope_sensor,
    laser_altimeter_sensor, laser_velocity_sensor,
    star_tracker_sensor, doppler_sensor, sat_range_tracker_sensor
)
import plotly.graph_objects as go
from lunanav.estimation.ekf import ekf_predict, ekf_update, Qd_from_accel_white, update_sensor_individual_NaN_check
from lunanav.sim.quaternion import unitize_state, angle_axis_to_q, quat_apply, conj
from lunanav.plotting import plot_control_effort, plot_state_vector_combined, plot_state_vector_combined_log, plot_state_vector
from lunanav.visualization import visualize_trajectory, plot_measurements, plot_attitude_relative_vertical, plot_filter_confidence, obsv_verbose, plot_sensor_config_comparison
from lunanav.sim.sensors import get_los_vectors
from lunanav.loaders import load_trajectory, load_sim_result, load_ekf_result
from lunanav.sim.generate import SatPosVel, make_sat_arrs, generate_env

In [ ]:

sim_result = load_sim_result("data/simresults/ilqr_better.json")

# s_true   = sim_result.s_true
# t        = sim_result.t_arr
dt       = sim_result.dt
n        = sim_result.nsteps
# mass_kg  = sim_result.mass_kg
# I        = sim_result.I.reshape((3, 3))

results = SimResults(n)

results.t = sim_result.t
results.states = sim_result.s_arr
results.force_N = sim_result.force
results.torque_Nm = sim_result.torque

lander = RigidBody(
    mass_kg=sim_result.mass_kg,
    I=sim_result.I
)

measurements_clean = {
    SensorName(k): np.array(v["truth"]) for k, v in sim_result.measurements.items()
}
measurements_noisy = {
    SensorName(k): np.array(v["noisy"]) for k, v in sim_result.measurements.items()
}
print(f"Loaded sim result: {n} steps, dt={dt}s")

In [ ]:


state0 = results.states[0]
sim = SimParams(state0, lander, dt, results.t[-1])

sensor_suite = SensorSuite(sensors={
    SensorName.ACCELEROMETER: accelerometer_sensor(sigma_accel),
    SensorName.GYROSCOPE: gyroscope_sensor(sigma_gyro),
    SensorName.LASER_ALTIMETER: laser_altimeter_sensor(sigma_los),
    SensorName.LASER_VELOCITY: laser_velocity_sensor(sigma_los_vel),
    SensorName.STAR_TRACKER: star_tracker_sensor(sigma_star),
    SensorName.DOPPLER: doppler_sensor(3, sigma_doppler),
    SensorName.RANGE_TRACKER: sat_range_tracker_sensor(3, sigma_sat_range_tracker),
})

doppler_sats = [
    make_sat_arrs(results.t, altitude=100e3, raan=0, aop=90, inc=90),
    make_sat_arrs(results.t, altitude=100e3, raan=90, aop=94, inc=94),
    make_sat_arrs(results.t, altitude=100e3, raan=160, aop=70, inc=86),
]

env_arr = generate_env(results, sim, doppler_sats)
print("Sensor suite and environment ready")

In [ ]:
from lunanav.estimation.ekf import ekf_predict, update_sensor, Qd_from_accel_white
from lunanav.sim.quaternion import unitize_state

# Q matrix
Q6 = Qd_from_accel_white(dt, sigma_accel)
Q_att = np.eye(4) * (sigma_gyro * dt)**2
Q_ang = np.eye(3) * (sigma_gyro * dt)**2
Q_ekf = np.block([
    [Q6,                   np.zeros((6, 4)), np.zeros((6, 3))],
    [np.zeros((4, 6)), Q_att,              np.zeros((4, 3))],
    [np.zeros((3, 6)), np.zeros((3, 4)), Q_ang],
])
print("EKF Q matrix ready")

# Sensor Ablation Study

Run EKF with different sensor subsets to evaluate observability.

In [ ]:
# Define sensor dropout configurations with meaningful combinations
sensor_configs = [
    {
        "name": "All Sensors",
        "description": "All sensors active (baseline)",
        "frequencies": {
            SensorName.LASER_ALTIMETER: 1,
            SensorName.LASER_VELOCITY: 1,
            SensorName.STAR_TRACKER: 1,
            SensorName.DOPPLER: 1,
            SensorName.RANGE_TRACKER: 1,
        }
    },
    # {
    #     "name": "No Altimeter",
    #     "description": "Height-blind configuration (no laser altitude)",
    #     "frequencies": {
    #         SensorName.LASER_ALTIMETER: None,
    #         SensorName.LASER_VELOCITY: 1,
    #         SensorName.STAR_TRACKER: 1,
    #         SensorName.DOPPLER: 1,
    #         SensorName.RANGE_TRACKER: 1,
    #     }
    # },
    # {
    #     "name": "No Laser Velocity",
    #     "description": "Without local velocity measurement",
    #     "frequencies": {
    #         SensorName.LASER_ALTIMETER: 1,
    #         SensorName.LASER_VELOCITY: None,
    #         SensorName.STAR_TRACKER: 1,
    #         SensorName.DOPPLER: 1,
    #         SensorName.RANGE_TRACKER: 1,
    #     }
    # },
    # {
    #     "name": "No Star Tracker",
    #     "description": "No absolute attitude reference",
    #     "frequencies": {
    #         SensorName.LASER_ALTIMETER: 1,
    #         SensorName.LASER_VELOCITY: 1,
    #         SensorName.STAR_TRACKER: None,
    #         SensorName.DOPPLER: 1,
    #         SensorName.RANGE_TRACKER: 1,
    #     }
    # },
    # {
    #     "name": "Laser Only",
    #     "description": "Terrain-relative navigation (ideal TRN)",
    #     "frequencies": {
    #         SensorName.LASER_ALTIMETER: 1,
    #         SensorName.LASER_VELOCITY: 1,
    #         SensorName.STAR_TRACKER: None,
    #         SensorName.DOPPLER: None,
    #         SensorName.RANGE_TRACKER: None,
    #     }
    # },
    # {
    #     "name": "Laser + Star Tracker",
    #     "description": "TRN with attitude reference",
    #     "frequencies": {
    #         SensorName.LASER_ALTIMETER: 1,
    #         SensorName.LASER_VELOCITY: 1,
    #         SensorName.STAR_TRACKER: 1,
    #         SensorName.DOPPLER: None,
    #         SensorName.RANGE_TRACKER: None,
    #     }
    # },
    # {
    #     "name": "Vision + Navigation",
    #     "description": "Star tracker + Doppler + Range (no local sensors)",
    #     "frequencies": {
    #         SensorName.LASER_ALTIMETER: None,
    #         SensorName.LASER_VELOCITY: None,
    #         SensorName.STAR_TRACKER: 1,
    #         SensorName.DOPPLER: 1,
    #         SensorName.RANGE_TRACKER: 1,
    #     }
    # },
    # {
    #     "name": "IMU Only",
    #     "description": "Dead reckoning (accelerometer + gyro only)",
    #     "frequencies": {
    #         SensorName.LASER_ALTIMETER: None,
    #         SensorName.LASER_VELOCITY: None,
    #         SensorName.STAR_TRACKER: None,
    #         SensorName.DOPPLER: None,
    #         SensorName.RANGE_TRACKER: None,
    #     }
    # },
]

sensor_frequencies = {
    SensorName.LASER_ALTIMETER: 1,
    SensorName.LASER_VELOCITY: 1,
    SensorName.STAR_TRACKER: 1,
    SensorName.DOPPLER: 1,
    SensorName.RANGE_TRACKER: 1,
}

In [ ]:
# offset = np.array([1e3, -1e3, 1e3, 120, 20, 46, *angle_axis_to_q(30, [0,1,0], True), 10 * DEG_TO_RAD, -20 * DEG_TO_RAD, 30 * DEG_TO_RAD])

seed = 178
rng = np.random.default_rng(seed)
offset = np.zeros(13)

# Position and velocity (first 6 elements)
offset = rng.multivariate_normal(np.zeros(13), Q_ekf * 100000)
print(f"Offset: {offset}")

In [ ]:
def run_configs(results: SimResults, configs: list[dict], measurements: dict, sensor_frequencies, offset = None):


    offset = offset if offset is not None else np.zeros(13)

    # Run EKF for each configuration (no initial offset)
    results_list = []

    for config in configs:
        print(f"\n{'='*60}")
        print(f"Running: {config['name']}")
        print(f"Description: {config['description']}")
        print(f"{'='*60}")
        
        mu_arr = np.zeros((n, 13)) * np.nan
        Sigma_arr = np.zeros((n, 13, 13)) * np.nan
        
        mu_arr[0] = results.states[0] + offset
        mu_arr[0] = unitize_state(mu_arr[0])
        Sigma_arr[0] = np.eye(13)
        
        valid = True
        for i in tqdm(range(n - 1)):
            # accel_meas = sim_result.measurements[SensorName.ACCELEROMETER.value]['truth'][i] 
            accel_meas = measurements[SensorName.ACCELEROMETER][i] 
            gyro_meas = results.states[i, 10:13]
            
            mu_pred, Sigma_pred = ekf_predict(mu_arr[i], Sigma_arr[i], accel_meas, gyro_meas, Q_ekf, sim)
            mu_pred = unitize_state(mu_pred)

            if jnp.any(jnp.isnan(mu_pred)):
                print(f"NaN after predict at i={i}")
                break
            
            env = env_arr[i]

            
            for sensor, use in config['frequencies'].items():
                if use in [0, None]:
                    continue
                
                freq = sensor_frequencies[sensor]
                if sensor in [SensorName.LASER_ALTIMETER, SensorName.LASER_VELOCITY]:
                    mu_pred, Sigma_pred = update_sensor_individual_NaN_check(sensor, freq, mu_pred, Sigma_pred, env, sensor_suite, measurements, i)
                else:
                    mu_pred, Sigma_pred = update_sensor(sensor, freq, mu_pred, Sigma_pred, env, sensor_suite, measurements, i)

                if jnp.any(jnp.isnan(mu_pred)):
                    print(f"NaN after update for {sensor} at i={i}")
                    valid = False
            mu_arr[i + 1] = mu_pred
            Sigma_arr[i + 1] = Sigma_pred

            if not valid:
                break
        
        pos_error = np.linalg.norm(mu_arr[:, 0:3] - results.states[:, 0:3], axis=1)
        vel_error = np.linalg.norm(mu_arr[:, 3:6] - results.states[:, 3:6], axis=1)
        att_error = np.linalg.norm(mu_arr[:, 6:10] - results.states[:, 6:10], axis=1)
        
        results_list.append({
            "name": config['name'],
            "description": config['description'],
            "config": config['frequencies'],
            "mu_arr": mu_arr,
            "Sigma_arr": Sigma_arr,
            "pos_error": pos_error,
            "vel_error": vel_error,
            "att_error": att_error,
        })

    print("\n" + "="*60)
    print("All configurations completed!")
    print("="*60)

    return results_list

In [ ]:
no_offset_results_list = run_configs(results, sensor_configs, measurements_noisy, sensor_frequencies)

In [ ]:
offset_results_list = run_configs(results, sensor_configs, measurements_noisy, sensor_frequencies, offset)

In [ ]:
def detect_dropout_times(measurements, t):
    """
    Detect times when sensors dropout (number -> NaN) or come back (NaN -> number).
    
    Parameters:
    -----------
    measurements : np.ndarray
        Measurement array (n_timesteps, n_sensors)
    t : np.ndarray
        Time array
    
    Returns:
    --------
    dropout_times : list
        Times when any sensor goes from valid to NaN
    undropout_times : list
        Times when any sensor goes from NaN to valid
    """
    n_valid = np.sum(~np.isnan(measurements), axis=1)  # Shape: (2000,)
    
    dropout_times = []
    undropout_times = []
    
    for i in range(1, len(n_valid)):
        # Number of valid sensors decreased
        if n_valid[i] < n_valid[i-1]:
            dropout_times.append(float(t[i]))
        # Number of valid sensors increased
        elif n_valid[i] > n_valid[i-1]:
            undropout_times.append(float(t[i]))
    
    return dropout_times, undropout_times

# Usage:
dropout_times, undropout_times = detect_dropout_times(
    measurements_noisy[SensorName.LASER_ALTIMETER], 
    results.t
)

print(f"Dropout times: {dropout_times}")
print(f"Undropout times: {undropout_times}")

# Plot with vertical lines at transitions
fig = go.Figure()
fig.add_trace(go.Scatter(x=results.t, y=measurements_noisy[SensorName.LASER_ALTIMETER][:,0], mode='lines', name='LOS1'))
fig.add_trace(go.Scatter(x=results.t, y=measurements_noisy[SensorName.LASER_ALTIMETER][:,1], mode='lines', name='LOS2'))
fig.add_trace(go.Scatter(x=results.t, y=measurements_noisy[SensorName.LASER_ALTIMETER][:,2], mode='lines', name='LOS3'))
fig.add_trace(go.Scatter(x=results.t, y=measurements_noisy[SensorName.LASER_ALTIMETER][:,3], mode='lines', name='LOS4'))

# Add vertical lines at dropout/undropout times
for t_drop in dropout_times:
    fig.add_vline(x=t_drop, line_dash="dash", line_color="red", annotation_text="Dropout")
for t_undrop in undropout_times:
    fig.add_vline(x=t_undrop, line_dash="dash", line_color="green", annotation_text="Return")

fig.show()

In [ ]:
# fig = go.Figure()
# fig.add_trace(go.Scatter(x=results.t, y=measurements_noisy[SensorName.LASER_ALTIMETER][:,0], mode='lines', name='LOS1'))
# fig.add_trace(go.Scatter(x=results.t, y=measurements_noisy[SensorName.LASER_ALTIMETER][:,1], mode='lines', name='LOS2'))
# fig.add_trace(go.Scatter(x=results.t, y=measurements_noisy[SensorName.LASER_ALTIMETER][:,2], mode='lines', name='LOS3'))
# fig.add_trace(go.Scatter(x=results.t, y=measurements_noisy[SensorName.LASER_ALTIMETER][:,3], mode='lines', name='LOS4'))
# fig.show()

# # dropout_indices = [7.8]
# # undropout_indices = [35.0, 73.1]



# dropout_times = []
# # undropout_times = [56.1, 64.7, 71.4, 86.6]
# undropout_times = [42.5, , 71.4, 86.6]

In [ ]:
MOON_3_VEC(1).flatten()

In [ ]:
visualize_trajectory([results.states, no_offset_results_list[0]["mu_arr"]], results.t, dt, offset = [0,0,R_MOON], title="EKF Estimated Trajectory with LOS Vectors", downsample_rate=5, moon_resolution = 35).show()

In [ ]:
plot_sensor_config_comparison(no_offset_results_list, results.t, dropout_times, undropout_times, alpha=0.8)

# Then plot for NO nan

In [ ]:
plot_sensor_config_comparison(offset_results_list, results.t, dropout_times, undropout_times, alpha=0.7)
